# EDA: 지층 구조 파악 / 분석

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, glob
from pathlib import Path

In [ ]:
DATA_DIR = Path("../data/raw")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

_train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
TRAIN_WIDS = [os.path.basename(f).split('__')[0] for f in _train_files]
_test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WIDS = [os.path.basename(f).split('__')[0] for f in _test_files]

print(f"훈련 데이터 총 우물 수: {len(TRAIN_WIDS)}")
print(f"훈련 데이터 우물 ID: {TRAIN_WIDS[:5]}")

def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw

훈련 데이터 총 우물 수: 773
훈련 데이터 우물 ID: ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']


In [20]:
# 우물별 지층 경계 통과 횟수
types = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
for wid in TRAIN_WIDS[:20]:
    hw, _ = load_well(wid, 'train')
    z = hw['Z'].values.astype(float)
    parts = []
    total_cross = 0
    for type in types:
        if type not in hw.columns:
            continue
        m = hw[type].values.astype(float)
        valid = ~np.isnan(m)
        if valid.sum() < 2:
            parts.append(f'{type}:NaN100%')
            continue
        s = np.sign(z[valid] - m[valid])
        cross = int((np.diff(s) != 0).sum())
        total_cross += cross
        parts.append(f'{type}:{(1-valid.mean()):.0%}NaN,{cross}회')
    print(f'{wid}  총통과={total_cross}  | ' + '  '.join(parts))


000d7d20  총통과=5  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,1회  EGFDL:0%NaN,1회  BUDA:0%NaN,0회
00bbac68  총통과=5  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,1회  EGFDL:0%NaN,1회  BUDA:0%NaN,0회
00e12e8b  총통과=4  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,2회  EGFDU:0%NaN,0회  EGFDL:0%NaN,0회  BUDA:0%NaN,0회
015fe0d2  총통과=5  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,1회  EGFDL:0%NaN,1회  BUDA:0%NaN,0회
01869cd4  총통과=5  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,1회  EGFDL:0%NaN,1회  BUDA:0%NaN,0회
01982c1d  총통과=3  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,0회  EGFDL:0%NaN,0회  BUDA:0%NaN,0회
028d7b28  총통과=6  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,4회  EGFDU:0%NaN,0회  EGFDL:0%NaN,0회  BUDA:0%NaN,0회
02e7fe5a  총통과=5  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,1회  EGFDL:0%NaN,1회  BUDA:0%NaN,0회
0390d174  총통과=3  | ANCC:0%NaN,1회  ASTNU:0%NaN,1회  ASTNL:0%NaN,1회  EGFDU:0%NaN,0회  EGFDL:0%NaN,0회  BUDA:0

In [22]:
# 지층 내에서 TVT = ref_tvt − (Z − marker) 형태인지 (구간별 선형성)
for wid in ['000d7d20', '01869cd4', '028d7b28']:   # 좋은('000d7d20')/나쁜('01869cd4', '028d7b28') 우물
    hw, _ = load_well(wid, 'train')
    types = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
    print(f'\n{wid}  TVT vs (Z − marker) 상관:')
    for type in types:
        if type in hw.columns and hw[type].notna().any():
            d = hw['Z'].values - hw[type].values
            v = ~np.isnan(d)
            r = np.corrcoef(d[v], hw['TVT'].values[v])[0,1]
            print(f'  {type}: corr={r:+.3f}')


000d7d20  TVT vs (Z − marker) 상관:
  ANCC: corr=-1.000
  ASTNU: corr=-1.000
  ASTNL: corr=-1.000
  EGFDU: corr=-1.000
  EGFDL: corr=-1.000
  BUDA: corr=-1.000

01869cd4  TVT vs (Z − marker) 상관:
  ANCC: corr=-1.000
  ASTNU: corr=-1.000
  ASTNL: corr=-1.000
  EGFDU: corr=-1.000
  EGFDL: corr=-1.000
  BUDA: corr=-1.000

028d7b28  TVT vs (Z − marker) 상관:
  ANCC: corr=-1.000
  ASTNU: corr=-1.000
  ASTNL: corr=-1.000
  EGFDU: corr=-1.000
  EGFDL: corr=-1.000
  BUDA: corr=-1.000


In [18]:
def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):   # 07에서 가져오기
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset

for wid in ['000d7d20', '01869cd4', '028d7b28']:
    hw, tw = load_well(wid, 'train')
    pred = tvt_from_contacts(hw, tw).values
    mask = hw['TVT_input'].isna().values
    r = np.sqrt(np.mean((hw['TVT'].values[mask] - pred[mask]) ** 2))
    print(f'{wid}  hidden 구간 contacts RMSE = {r:.4f}')


000d7d20  hidden 구간 contacts RMSE = 0.0053
01869cd4  hidden 구간 contacts RMSE = 0.0056
028d7b28  hidden 구간 contacts RMSE = 0.0057


In [23]:
# 각 점이 어느 지층인지 라벨 만들고, 지층별 GR 분포 비교
wid = '000d7d20'
hw, _ = load_well(wid, 'train')
markers = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
M = hw[markers].values                       # 각 점에서 지층 경계 깊이들
z = hw['Z'].values[:, None]
layer = (z > M).sum(1)                        # 지층 인덱스
hw = hw.assign(layer=layer)
print(hw.groupby('layer')['GR'].agg(['mean','std','count']))


             mean        std  count
layer                              
1       96.394019  15.838581   2526
2       65.754950  10.864538     52
3       52.689466  18.787205    104
4       84.547636  13.213359     40
5       89.968225  16.566653    189
6      113.976794  16.757018    109


# EDA 요약

## 1. TVT의 정의
- `TVT = 지층 기준값 − (Z − 지층경계 type)`,  **corr = −1.000**
- -> 지층(type)만 알면 TVT가 자동 계산됨
- train엔 type 있음 / **test엔 없음** -> **[핵심] GR로 지층을 예측해야함**

## 2. 데이터 구조
- 우물당 지층 **3~6회 통과**
- 지층별 GR 구분됨(53~114) — 겹침도 있어서 단일 점이 아닌 **시퀀스**로 봐야 함

## 3. PF/LSTM의 한계
- 드릴이 **한 층에만** 있다고(TVT가 연속이라고) 가정
- 실제 TVT는 경계에서 **점프** -> 가정에 위배돼서 큰 오차가 생김

## 4. 해결 방향 — Viterbi
- 지층(이산 상태)을 추적
